# AIDA 1141 Project Exercise 04 – Routing Label Definition

**Question:** How does each synthetic assessment receive its risk label, and could any selected model input expose the answer?

This notebook uses the original request, assessment, status-event, and simulation files from Exercise 02. It checks label consistency and prediction-time boundaries, examines strong input patterns, and creates a small labelled sample for review.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

EX02 = Path("../02-synthetic-request-records")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

read = lambda name: pd.read_csv(EX02 / "data" / name, keep_default_na=False, na_values=[""])
requests, assessments, events, truth = (read(f) for f in
    ["requests.csv", "assessments.csv", "status_events.csv", "simulation_truth.csv"])
dictionary = pd.read_csv(EX02 / "data-dictionary.csv", keep_default_na=False)
for df, cols in [(requests, ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at"]),
                 (assessments, ["assessed_at"]), (events, ["event_at"])]:
    for c in cols:
        df[c] = pd.to_datetime(df[c])

checks = []  # (check, result, passed)
print(f"{len(requests):,} requests | {len(assessments):,} assessments")

2,064 requests | 2,579 assessments


In [2]:
# The label: would the task miss 08:00 on the start date without intervention (no-intervention simulation)
def misses_deadline(completed_at, start_date):
    """At risk if the team finishes after 08:00 on the start date; finishing exactly at 08:00 counts as ready."""
    return completed_at > pd.to_datetime(start_date) + pd.Timedelta(hours=8)

# Constructed boundary test: just before, exactly at, and just after the deadline
boundary = [misses_deadline(pd.Timestamp(t), "2026-03-16") for t in
            ["2026-03-16 07:59", "2026-03-16 08:00", "2026-03-16 08:01"]]
checks.append(("deadline rule: 07:59 ready, 08:00 ready, 08:01 at risk", "constructed test",
               boundary == [False, False, True]))

# Recompute the label and the overdue flag independently from timestamps
by_req = requests.set_index("request_id")
completed = assessments["request_id"].map(by_req["team_completed_at"])
start = pd.to_datetime(assessments["request_id"].map(by_req["confirmed_start_date"]))
deadline_at = start + pd.Timedelta(hours=8)
checks += [
    ("at_risk recomputed from completion time and 08:00 deadline", "all assessments",
     bool((misses_deadline(completed, start) == assessments["at_risk"]).all())),
    ("past_deadline_at_assessment recomputed from assessment time", "all assessments",
     bool(((assessments["assessed_at"] >= deadline_at) == assessments["past_deadline_at_assessment"]).all())),
    ("label is the same at both checkpoints of a task", "all tasks",
     bool(assessments.groupby("request_id")["at_risk"].nunique().eq(1).all())),
]

early = assessments[~assessments["past_deadline_at_assessment"]]
overdue = assessments[assessments["past_deadline_at_assessment"]]
label_summary = pd.DataFrame({
    "assessments": [len(assessments), len(early), len(overdue)],
    "at_risk": [assessments["at_risk"].sum(), early["at_risk"].sum(), overdue["at_risk"].sum()],
}, index=["all", "early-warning", "already overdue"])
label_summary["at_risk_%"] = (label_summary["at_risk"] / label_summary["assessments"] * 100).round(1)
label_summary

,assessments,at_risk,at_risk_%
all,2579,471,18.3
early-warning,2516,408,16.2
already overdue,63,63,100.0


## Labels and staff decisions

The model has two labels: `at_risk = True` when the simulated task finishes after 08:00 on the confirmed start date, and `False` when it finishes at or before that deadline. Labels come from the no-intervention simulation, not real staff decisions. Both checkpoints for a task share the same outcome label.

The workflow’s four actions are separate from these labels. Rules identify the responsible team and missing information, while the model will suggest whether follow-up is needed. The coordinator confirms routing, decides whether to follow up or expedite, and reviews uncertain or unusual cases.

The proposed 90% recall target applies only to assessments made before the deadline. Assessments at or after the deadline are reported separately because they identify an existing delay rather than provide early warning. Further definitions are documented in [label-guide.md](label-guide.md).

In [3]:
# Leakage check 1: model inputs come only from fields with role "model_input" in the data dictionary
roles = dictionary.drop_duplicates("field").set_index("field")["role"]
MODEL_INPUTS = list(dictionary.loc[dictionary["role"] == "model_input", "field"])
excluded = ["at_risk", "ready_by_start", "team_completed_at", "completion_confirmed_at",
            "past_deadline_at_assessment", "sim_base_processing_days", "sim_workload_delay_days",
            "sim_hidden_delay_days", "sim_delay_reason"]
overlap = set(MODEL_INPUTS) & set(excluded)
checks.append(("no label, outcome, or simulation-only field is a model input",
               f"{len(MODEL_INPUTS)} inputs checked", not overlap))
pd.DataFrame({"field": MODEL_INPUTS + excluded,
              "role": [roles[f] for f in MODEL_INPUTS + excluded],
              "used_as_input": [True] * len(MODEL_INPUTS) + [False] * len(excluded)})

,field,role,used_as_input
0,task_type,model_input,True
1,responsible_team,model_input,True
2,hiring_department,model_input,True
3,employee_type,model_input,True
4,intake_channel,model_input,True
5,info_issue_at_intake,model_input,True
6,business_days_until_start,model_input,True
7,current_status,model_input,True
8,business_days_in_current_status,model_input,True
9,team_open_workload,model_input,True


In [4]:
# Leakage check 2: every input is computable from information available at assessment time
checks += [
    ("every assessment happens at or after information is complete", "all assessments",
     bool((assessments["assessed_at"] >= assessments["request_id"].map(by_req["info_complete_at"])).all())),
    ("every assessment happens before the team finishes", "all assessments",
     bool((assessments["assessed_at"] < completed).all())),
]

# Recompute days until start from the assessment date and the confirmed start date only
recomputed = np.busday_count(assessments["assessed_at"].values.astype("datetime64[D]"),
                             start.values.astype("datetime64[D]"))
checks.append(("business_days_until_start recomputed from assessment date and confirmed start date",
               "all assessments", bool((recomputed == assessments["business_days_until_start"]).all())))

# Recompute team workload from the status event log, counting events at or before the assessment
assigned = events[events["event_type"] == "assigned_to_team"].set_index("request_id")["event_at"]
finished = events[events["event_type"] == "team_completed"].set_index("request_id")["event_at"]
timeline = pd.DataFrame({"team": by_req["responsible_team"], "assigned": assigned, "finished": finished})
def workload_at(row):
    t = timeline[timeline["team"] == by_req.at[row["request_id"], "responsible_team"]]
    return int(((t["assigned"] <= row["assessed_at"]) & (t["finished"] > row["assessed_at"])).sum())
recomputed_workload = assessments.apply(workload_at, axis=1)
checks.append(("team_open_workload recomputed from events at or before the assessment", "all assessments",
               bool((recomputed_workload == assessments["team_open_workload"]).all())))
pd.DataFrame(checks, columns=["check", "scope", "passed"])

,check,scope,passed
0,"deadline rule: 07:59 ready, 08:00 ready, 08:01...",constructed test,True
1,at_risk recomputed from completion time and 08...,all assessments,True
2,past_deadline_at_assessment recomputed from as...,all assessments,True
3,label is the same at both checkpoints of a task,all tasks,True
4,"no label, outcome, or simulation-only field is...",10 inputs checked,True
5,every assessment happens at or after informati...,all assessments,True
6,every assessment happens before the team finishes,all assessments,True
7,business_days_until_start recomputed from asse...,all assessments,True
8,team_open_workload recomputed from events at o...,all assessments,True


In [5]:
# Inspect strong single-input patterns (early-warning only); strength alone does not establish leakage
inputs = early.merge(requests[["request_id", "task_type", "responsible_team", "hiring_department",
                               "employee_type", "intake_channel", "info_issue_at_intake"]], on="request_id")
rows = []
for f in ["task_type", "responsible_team", "hiring_department", "employee_type", "intake_channel",
          "info_issue_at_intake", "current_status"]:
    for value, g in inputs.groupby(f):
        rows.append({"input": f, "group": str(value), "assessments": len(g), "at_risk_%": round(g["at_risk"].mean() * 100, 1)})
bins = {"business_days_until_start": [-np.inf, 2, 5, 10, 20, np.inf],
        "business_days_in_current_status": [-np.inf, 0, 2, 5, np.inf],
        "team_open_workload": [-np.inf, 5, 10, 20, np.inf]}
for f, edges in bins.items():
    for value, g in inputs.groupby(pd.cut(inputs[f], edges, include_lowest=True), observed=True):
        rows.append({"input": f, "group": str(value), "assessments": len(g), "at_risk_%": round(g["at_risk"].mean() * 100, 1)})
separation = pd.DataFrame(rows)
separation.to_csv(OUTPUT_DIR / "single_input_risk_rates.csv", index=False)
print("Groups with at least 30 assessments that are at least 95% or at most 5% at risk:")
separation[(separation["assessments"] >= 30) & ((separation["at_risk_%"] >= 95) | (separation["at_risk_%"] <= 5))]

Groups with at least 30 assessments that are at least 95% or at most 5% at risk:


,input,group,assessments,at_risk_%
15,employee_type,permanent,1193,3.1
25,business_days_until_start,"(-inf, 2.0]",122,99.2
28,business_days_until_start,"(10.0, 20.0]",648,0.5
29,business_days_until_start,"(20.0, inf]",440,0.0
33,business_days_in_current_status,"(5.0, inf]",44,2.3


## Leakage checks and strong predictors

The checks examine whether labels agree with simulated completion times, whether assessments occur after information is complete and before team completion, and whether days until start and team workload can be independently recomputed. The selected feature list also excludes labels, outcomes, and simulation-only fields.

These checks provide evidence that the inputs respect the assessment-time boundary, but they do not establish that all possible leakage is absent.

A strong predictor is not automatically leakage. Days until start and employee type are known at assessment time, and their relationship with lateness reflects the generator’s assumptions. These patterns make comparison with a simple baseline important in the next exercise.

In [6]:
# Edge cases
deadline = pd.to_datetime(by_req["confirmed_start_date"]) + pd.Timedelta(hours=8)
merged = assessments.merge(requests[["request_id", "info_issue_at_intake"]], on="request_id")
edge_cases = pd.DataFrame([
    ("already overdue at assessment (always at risk)", len(overdue), int(overdue["at_risk"].sum())),
    ("team finished exactly at the 08:00 deadline (counts as ready)",
     int((by_req["team_completed_at"] == deadline).sum()), 0),
    ("information issue at intake and at risk", int((merged["info_issue_at_intake"] & merged["at_risk"]).sum()),
     int((merged["info_issue_at_intake"] & merged["at_risk"]).sum())),
    ("checkpoint 2 (only tasks still unfinished)", int((assessments["checkpoint"] == "checkpoint_2").sum()),
     int(assessments.loc[assessments["checkpoint"] == "checkpoint_2", "at_risk"].sum())),
    ("late outcome despite 10+ business days at assessment",
     int(((early["business_days_until_start"] >= 10) & early["at_risk"]).sum()),
     int(((early["business_days_until_start"] >= 10) & early["at_risk"]).sum())),
], columns=["edge case", "assessments_or_tasks", "at_risk"])
edge_cases

,edge case,assessments_or_tasks,at_risk
0,already overdue at assessment (always at risk),63,63
1,team finished exactly at the 08:00 deadline (c...,0,0
2,information issue at intake and at risk,78,78
3,checkpoint 2 (only tasks still unfinished),515,77
4,late outcome despite 10+ business days at asse...,11,11


## Edge cases and handling

Assessments at or after the start deadline are labelled at risk because the task is still unfinished. They are excluded from the early-warning recall target and reported separately.

Completion exactly at 08:00 counts as ready. The constructed boundary test checks completion at 07:59, 08:00, and 08:01, with expected risk labels of False, False, and True. This tests the rule even if no exact-deadline example occurs in the generated records.

An information issue at intake does not mean information is still missing during assessment. The initial issue has already been resolved before checkpoint 1, so it represents the request’s history.

Checkpoint 2 includes only tasks that remain unfinished, making it a selected group. A task that finishes late despite substantial lead time may reflect workload, processing duration, or hidden delays; the timing alone does not identify the cause.

In [7]:
# Labelled sample: a small, stratified set covering every case (simulation fields used only to explain)
sample_src = (assessments
              .merge(requests[["request_id", "task_type", "responsible_team", "hiring_department", "employee_type",
                               "intake_channel", "info_issue_at_intake", "confirmed_start_date", "team_completed_at"]],
                     on="request_id")
              .merge(truth, on="request_id"))
s = sample_src
strata = {
    "at risk – checkpoint 1": s["at_risk"] & (s["checkpoint"] == "checkpoint_1") & ~s["past_deadline_at_assessment"],
    "at risk – checkpoint 2": s["at_risk"] & (s["checkpoint"] == "checkpoint_2"),
    "at risk – hidden delay": s["at_risk"] & (s["sim_hidden_delay_days"] > 0) & ~s["past_deadline_at_assessment"],
    "already overdue": s["past_deadline_at_assessment"],
    "info issue and at risk": s["info_issue_at_intake"] & s["at_risk"] & ~s["past_deadline_at_assessment"],
    "not at risk – checkpoint 1": ~s["at_risk"] & (s["checkpoint"] == "checkpoint_1"),
    "not at risk – checkpoint 2": ~s["at_risk"] & (s["checkpoint"] == "checkpoint_2"),
    "not at risk – tight timeline (≤5 days)": ~s["at_risk"] & (s["business_days_until_start"] <= 5),
    "info issue but not at risk": s["info_issue_at_intake"] & ~s["at_risk"],
    "late despite 10+ business days left": s["at_risk"] & (s["business_days_until_start"] >= 10),
}
picked, used = [], set()
for case, mask in strata.items():
    pool = s[mask & ~s["assessment_id"].isin(used)]
    take = pool.sample(n=min(2, len(pool)), random_state=2026)
    used.update(take["assessment_id"])
    picked.append(take.assign(case=case))
sample = pd.concat(picked, ignore_index=True)

def explain(r):
    if r["past_deadline_at_assessment"]:
        return "assessed at or after the 08:00 deadline"
    parts = [f"total base processing {r['sim_base_processing_days']} days"]
    if r["sim_hidden_delay_days"]: parts.append(f"hidden {r['sim_delay_reason'].replace('_', ' ')} (+{r['sim_hidden_delay_days']} days)")
    if r["sim_workload_delay_days"]: parts.append(f"busy team (+{r['sim_workload_delay_days']} days)")
    if r["info_issue_at_intake"]: parts.append("time lost resolving intake information")
    parts.append(f"{r['business_days_until_start']} business days left at assessment")
    return ("finished late: " if r["at_risk"] else "finished on time: ") + "; ".join(parts)

sample["reference_next_step_not_a_prediction"] = np.where(sample["at_risk"], "follow up / expedite", "proceed normally")
sample["explanation_not_an_input"] = sample.apply(explain, axis=1)
columns = ["case", "assessment_id", "request_id", "checkpoint"] + MODEL_INPUTS + \
          ["at_risk", "reference_next_step_not_a_prediction", "explanation_not_an_input"]
sample[columns].to_csv(OUTPUT_DIR / "labelled_sample.csv", index=False)
print(f"{len(sample)} labelled examples saved")
sample[["case", "checkpoint", "business_days_until_start", "team_open_workload", "at_risk", "explanation_not_an_input"]]

20 labelled examples saved


,case,checkpoint,business_days_until_start,team_open_workload,at_risk,explanation_not_an_input
0,at risk – checkpoint 1,checkpoint_1,3,9,True,finished late: total base processing 6 days; 3...
1,at risk – checkpoint 1,checkpoint_1,6,16,True,finished late: total base processing 4 days; h...
2,at risk – checkpoint 2,checkpoint_2,5,16,True,finished late: total base processing 5 days; 5...
3,at risk – checkpoint 2,checkpoint_2,5,5,True,finished late: total base processing 4 days; h...
4,at risk – hidden delay,checkpoint_1,6,38,True,finished late: total base processing 4 days; h...
5,at risk – hidden delay,checkpoint_1,2,14,True,finished late: total base processing 2 days; h...
6,already overdue,checkpoint_1,0,10,True,assessed at or after the 08:00 deadline
7,already overdue,checkpoint_1,-1,3,True,assessed at or after the 08:00 deadline
8,info issue and at risk,checkpoint_1,5,21,True,finished late: total base processing 5 days; t...
9,info issue and at risk,checkpoint_1,5,22,True,finished late: total base processing 4 days; h...


In [8]:
pd.DataFrame(checks, columns=["check", "scope", "passed"]).to_csv(OUTPUT_DIR / "leakage_checks.csv", index=False)
print("All checks passed:", all(c[2] for c in checks))

All checks passed: True


## What the labelled sample shows

The sample deliberately includes different labels, checkpoints, intake histories, and simulated delay conditions. Its reference next step is derived from the outcome label and is not a model prediction. Explanations may use simulation-only information for auditing, but that information is excluded from model inputs.

The sample illustrates how labels are assigned rather than representing the dataset’s class proportions or model performance. Total processing durations in the explanations describe the whole task, not the remaining work at checkpoint 2.

Labels are accurate to the simulated outcomes, but those outcomes depend on invented assumptions. Hidden delays create uncertainty from the model’s perspective rather than incorrect labels. Later evaluation will keep all records for each fictional employee together when splitting training, validation, and test data.